# Marimba quickstart on Google Colab

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Charcot-AI/marimba-api/blob/main/notebooks/colab_quickstart.ipynb)

This notebook installs the published `marimba-audio` 0.1.0 package and runs its real model on public LibriSpeech audio. Speech and turn-end questions use one validation utterance with 0.3 seconds of trailing silence. Speaker-change and people-count use two concatenated LibriSpeech samples from different speakers; the small requested validation split contains only one speaker, so the first 1.25 seconds of a second public LibriSpeech sample supplies the other voice. The speaker-change readout compares the first 6 seconds with the last 2 seconds of the final 8-second window, so the change is placed in the last 2 seconds. The transcripts are printed so the term choices can be checked. These examples demonstrate the API, not model accuracy.

The PyPI `torch` extra pins Torch and TorchAudio to 2.8.0. Colab's CUDA build with that base version satisfies the pins; the device check below reports whether CUDA remains available. A Hugging Face token is only needed while the model weights are private. In Colab, add it as a notebook secret named `HF_TOKEN`; never paste it into a cell.

In [1]:
!pip install -q "marimba-audio[torch]==0.1.0" datasets

In [2]:
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="enable_nested_tensor is True.*")

In [3]:
import io
import os
import re
import time

import numpy as np
import soundfile as sf
import torch
from datasets import Audio, load_dataset

from marimba import Marimba, Question

try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("Hugging Face token available; value is not displayed.")
else:
    print("No HF_TOKEN found. A token is only needed while the model weights are private.")

print(f"PyTorch {torch.__version__}; CUDA runtime: {torch.version.cuda}")
if torch.cuda.is_available():
    device = "cuda"
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    print("CUDA is unavailable; falling back to CPU.")

if device == "cpu":
    torch.set_num_threads(4)

sample_rate = 16_000
utterance_dataset = load_dataset(
    "hf-internal-testing/librispeech_asr_dummy", "clean", split="validation"
).cast_column("audio", Audio(sampling_rate=sample_rate, decode=False))
utterance = utterance_dataset[0]
speech_audio, audio_sample_rate = sf.read(io.BytesIO(utterance["audio"]["bytes"]), dtype="float32")
assert audio_sample_rate == sample_rate

speaker_dataset = load_dataset(
    "sanchit-gandhi/librispeech_asr_dummy", "default", split="validation.other"
).cast_column("audio", Audio(sampling_rate=sample_rate, decode=False))
second_speaker = next(
    row for row in speaker_dataset if row["speaker_id"] != utterance["speaker_id"]
)
second_audio, second_sample_rate = sf.read(
    io.BytesIO(second_speaker["audio"]["bytes"]), dtype="float32"
)
assert second_sample_rate == sample_rate
second_audio = second_audio[: int(1.25 * sample_rate)]

transcript = utterance["text"]
transcript_words = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", transcript)
normalized_words = {word.casefold() for word in transcript_words}
present_term = max(transcript_words, key=len).casefold()
absent_term = next(
    word for word in ("marimba", "quasar", "xylophone") if word not in normalized_words
)
assert present_term in normalized_words
assert absent_term not in normalized_words

tail_s = 0.3
tail = np.zeros(int(tail_s * sample_rate), dtype=np.float32)
utterance_audio = np.concatenate((speech_audio, tail))
speaker_audio = np.concatenate((speech_audio, tail, second_audio, tail))
print(f"Utterance speaker {utterance['speaker_id']} transcript: {transcript}")
print(
    f"Speech and turn-end clip: {len(utterance_audio) / sample_rate:.2f}s, "
    f"including {tail_s:.1f}s trailing silence."
)
print(f"Second speaker {second_speaker['speaker_id']} transcript: {second_speaker['text']}")
print(f"Speaker-change and people-count clip: {len(speaker_audio) / sample_rate:.2f}s")
print(f"Term in transcript: {present_term}; term absent from transcript: {absent_term}")

model = Marimba.from_pretrained(backend="torch", device=device, dtype="float32")

Hugging Face token available; value is not displayed.
PyTorch 2.8.0+cu128; CUDA runtime: 12.8
CUDA is unavailable; falling back to CPU.


Utterance speaker 1272 transcript: MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL
Speech and turn-end clip: 6.16s, including 0.3s trailing silence.
Second speaker 3660 transcript: GERAINT AS HE HAD BEEN USED TO DO WHEN HE WAS AT ARTHUR'S COURT FREQUENTED TOURNAMENTS
Speaker-change and people-count clip: 7.71s
Term in transcript: quilter; term absent from transcript: marimba


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [4]:
utterance_questions = [
    Question.noul("Is someone speaking?"),
    Question.score("Has the speaker finished?"),
    Question.choice(
        "What language is being spoken?", ["English", "Portuguese", "Spanish"], task="language"
    ),
    Question.noul(f'Did someone say "{present_term}"?'),
    Question.noul(f'Did someone say "{absent_term}"?'),
]
speaker_questions = [
    Question.score("Did the speaker change?"),
    Question.choice("How many people are talking?", ["0", "1", "2", "3+"]),
]


def run_call(label, audio_input, questions):
    if device == "cuda":
        torch.cuda.synchronize()
    started = time.perf_counter()
    decisions = model(audio_input, questions, sampling_rate=sample_rate)
    if device == "cuda":
        torch.cuda.synchronize()
    latency_ms = (time.perf_counter() - started) * 1000
    print({"call": label, "device": device, "call_latency_ms": round(latency_ms, 2)})
    for decision in decisions:
        print(decision.to_dict())
    return latency_ms


_ = model(utterance_audio, utterance_questions, sampling_rate=sample_rate)
utterance_latency_ms = run_call("utterance", utterance_audio, utterance_questions)
speaker_latency_ms = run_call("two_speakers", speaker_audio, speaker_questions)

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


{'call': 'utterance', 'device': 'cpu', 'call_latency_ms': 6863.82}
{'question': {'text': 'Is someone speaking?', 'kind': 'noul', 'options': None, 'task': 'voice_activity', 'threshold': None, 'seconds': None}, 'task': 'voice_activity', 'kind': 'noul', 'value': True, 'probability': 0.9829917913702354, 'probabilities': None, 'score': 2.65346097946167, 'threshold': 0.225, 'window_start_s': 0.0, 'window_end_s': 6.155, 'latency_ms': 6863.623521000022, 'status': 'validated', 'calibrated': False}
{'question': {'text': 'Has the speaker finished?', 'kind': 'score', 'options': None, 'task': 'turn_end', 'threshold': None, 'seconds': None}, 'task': 'turn_end', 'kind': 'score', 'value': 0.992970500312869, 'probability': 0.992970500312869, 'probabilities': None, 'score': 0.9919405998673774, 'threshold': 0.5, 'window_start_s': 0.0, 'window_end_s': 6.155, 'latency_ms': 6863.623521000022, 'status': 'validated', 'calibrated': False}
{'question': {'text': 'What language is being spoken?', 'kind': 'choice'

{'call': 'two_speakers', 'device': 'cpu', 'call_latency_ms': 266.73}
{'question': {'text': 'Did the speaker change?', 'kind': 'score', 'options': None, 'task': 'speaker_change', 'threshold': None, 'seconds': None}, 'task': 'speaker_change', 'kind': 'score', 'value': None, 'probability': None, 'probabilities': None, 'score': None, 'threshold': 0.5, 'window_start_s': 0.0, 'window_end_s': 7.705, 'latency_ms': 266.52026900001147, 'status': 'validated', 'calibrated': False}
{'question': {'text': 'How many people are talking?', 'kind': 'choice', 'options': ['0', '1', '2', '3+'], 'task': 'speaker_count', 'threshold': None, 'seconds': None}, 'task': 'speaker_count', 'kind': 'choice', 'value': None, 'probability': None, 'probabilities': None, 'score': None, 'threshold': 0.5, 'window_start_s': 0.0, 'window_end_s': 7.705, 'latency_ms': 266.52026900001147, 'status': 'validated', 'calibrated': False}


## Measured latency

The first call uses the same utterance and questions and is excluded as untimed warmup.

- CPU (four Torch threads), utterance call: **6,863.82 ms**
- CPU (four Torch threads), two-speaker call: **266.73 ms**
- Colab free T4 GPU: **[PENDING]**